In [1]:
import pandas as pd
import re

# 1. Load raw dataset using raw string path
raw_path = r'C:\Users\pietr\Desktop\IronHack\TeamBProject\GSAF5.csv'
df_raw = pd.read_csv(raw_path)

# 2. Select required columns
selected_cols = ['Date', 'Year', 'Country', 'State', 'Location', 'Fatal Y/N', 'Time']
df = df_raw[selected_cols].copy()

# 3. Standardize column names (lowercase, underscores)
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.replace('?', '')
)

# 4. Standardize text formatting: strip spaces and convert strings to lower
text_cols = ['country', 'state', 'location']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
    # Convert text representations of missing data to actual NA
    df[col] = df[col].replace(['nan', 'none', 'unknown', '?'], pd.NA)

# 5. Clean 'fatal_y_n': fill missing/invalid values with 'n'
df['fatal_y_n'] = df['fatal_y_n'].astype(str).str.strip().str.lower()
df['fatal_y_n'] = df['fatal_y_n'].apply(lambda x: x if x in ['y', 'n'] else 'n')

# 6. Helper to parse time strings into total minutes past midnight
def parse_time_to_minutes(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip().lower()
    digits = re.sub(r'\D', '', val_str)
    
    if len(digits) == 4:
        hrs, mins = int(digits[:2]), int(digits[2:])
        if 0 <= hrs < 24 and 0 <= mins < 60:
            return hrs * 60 + mins
    elif len(digits) == 3:
        hrs, mins = int(digits[0]), int(digits[1:])
        if 0 <= hrs < 24 and 0 <= mins < 60:
            return hrs * 60 + mins
    return None

# Calculate average time in minutes across valid entries
minutes_series = df['time'].apply(parse_time_to_minutes)
avg_minutes = int(minutes_series.dropna().mean())
avg_hrs = avg_minutes // 60
avg_mins = avg_minutes % 60
avg_time_str = f"{avg_hrs:02d}:{avg_mins:02d}"

print(f"Calculated Average Time for missing values: {avg_time_str}")

# Function to assign standard HH:MM time or average time
def clean_time_with_average(val):
    mins = parse_time_to_minutes(val)
    if mins is not None:
        hrs = mins // 60
        m = mins % 60
        return f"{hrs:02d}:{m:02d}"
    return avg_time_str

df['time'] = df['time'].apply(clean_time_with_average)

# 7. Clean 'year' column and drop invalid/missing rows
df['year'] = df['year'].astype(str).str.replace(',', '.').str.strip()
df['year'] = pd.to_numeric(df['year'], errors='coerce')

df_clean = df.dropna(subset=['country', 'state', 'location', 'year']).copy()
df_clean['year'] = df_clean['year'].astype(int)

# 8. Verify missing values summary
print("\n--- Cleaned DataFrame Summary ---")
print(f"Total rows remaining: {len(df_clean)}")
print(f"Remaining null values:\n{df_clean.isnull().sum()}")

# 9. Re-save to pickle file
df_clean.to_pickle('cleaned_shark_attacks.pkl')
print("\n✅ Dataset saved successfully to 'cleaned_shark_attacks.pkl'")

Calculated Average Time for missing values: 13:17

--- Cleaned DataFrame Summary ---
Total rows remaining: 6315
Remaining null values:
date         0
year         0
country      0
state        0
location     0
fatal_y_n    0
time         0
dtype: int64

✅ Dataset saved successfully to 'cleaned_shark_attacks.pkl'


In [2]:

import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Load the cleaned DataFrame from pickle
df_app = pd.read_pickle('cleaned_shark_attacks.pkl')

# 2. UI Components Setup
search_box = widgets.Text(
    value='',
    placeholder='Type a location (e.g., Sorrento, Florida, Hawaii)...',
    description='Location:',
    layout=widgets.Layout(width='420px')
)

search_button = widgets.Button(
    description="Check Safety",
    button_style='primary',
    icon='shield'
)

output_area = widgets.Output()


# 3. Term-Splitting Safety & Validation Function
def check_beach_safety_pure_python(b):
    with output_area:
        clear_output()
        
        raw_input = search_box.value.strip()
        if not raw_input:
            print("⚠️ Please enter a beach, city, state, or country name.")
            return

        # -------------------------------------------------------------
        # STEP 1: TERM SPLITTING & CLEANING
        # -------------------------------------------------------------
        # Replace punctuation and split text into individual words
        clean_text = raw_input.replace(',', ' ').replace('.', ' ').replace('-', ' ')
        words = clean_text.split()
        
        # Keep terms with more than 2 letters (filters out words like 'in', 'at', 'of')
        search_terms = [w.lower().strip() for w in words if len(w.strip()) > 2]

        if not search_terms:
            print("⚠️ Please enter a longer search term.")
            return

        # -------------------------------------------------------------
        # STEP 2: VERIFY VALID TERMS IN DATASET
        # -------------------------------------------------------------
        valid_terms = []
        for term in search_terms:
            term_matches = df_app[
                df_app['location'].fillna('').str.lower().str.contains(term) |
                df_app['state'].fillna('').str.lower().str.contains(term) |
                df_app['country'].fillna('').str.lower().str.contains(term)
            ]
            if len(term_matches) > 0:
                valid_terms.append(term)

        # Rejection: If none of the typed terms match any record in the dataset
        if not valid_terms:
            print(f"❌ Unknown Location: '{raw_input}'")
            print("We couldn't find this location or any matching region in our database.")
            print("Please check your spelling or try searching a broader region (e.g., state or country).")
            return

        # -------------------------------------------------------------
        # STEP 3: FILTER DATASET WITH RECOGNIZED TERMS
        # -------------------------------------------------------------
        regex_query = "|".join(valid_terms)
        matches = df_app[
            df_app['location'].fillna('').str.lower().str.contains(regex_query) |
            df_app['state'].fillna('').str.lower().str.contains(regex_query) |
            df_app['country'].fillna('').str.lower().str.contains(regex_query)
        ]

        total_incidents = len(matches)
        fatal_incidents = len(matches[matches['fatal_y_n'] == 'Y'])

        # -------------------------------------------------------------
        # STEP 4: CALCULATE SAFETY GRADE (S to F)
        # -------------------------------------------------------------
        if total_incidents == 0:
            grade, status = 'S', 'Safe (No historical incidents recorded)'
        elif total_incidents == 1 and fatal_incidents == 0:
            grade, status = 'A', 'Very Low Risk'
        elif total_incidents <= 3 and fatal_incidents == 0:
            grade, status = 'B', 'Low Risk'
        elif fatal_incidents <= 1 and total_incidents <= 5:
            grade, status = 'C', 'Moderate Risk'
        elif fatal_incidents > 1 or total_incidents > 5:
            grade, status = 'F', 'High Risk Spot'
        else:
            grade, status = 'D', 'Elevated Risk'

        # -------------------------------------------------------------
        # STEP 5: DISPLAY REPORT
        # -------------------------------------------------------------
        print("==================================================")
        print(f"📊 SAFETY REPORT: {raw_input.title()}")
        print("==================================================")
        print(f"🔎 Recognized Location Terms: {', '.join(valid_terms).title()}")
        print(f"🛡️ Safety Grade: {grade} ({status})")
        print(f"🦈 Total Recorded Incidents: {total_incidents}")
        print(f"⚠️ Fatal Incidents: {fatal_incidents}")
        print("--------------------------------------------------\n")

        if total_incidents > 0:
            print("📋 Preview of Historical Records:")
            display(matches[['date', 'year', 'country', 'state', 'location', 'fatal_y_n', 'time']].head(5))


# 4. Attach Event and Display Widgets
search_button.on_click(check_beach_safety_pure_python)
display(search_box, search_button, output_area)

Text(value='', description='Location:', layout=Layout(width='420px'), placeholder='Type a location (e.g., Sorr…

Button(button_style='primary', description='Check Safety', icon='shield', style=ButtonStyle())

Output()